<a href="https://colab.research.google.com/github/AliAI11/DolphinMind/blob/main/notebooks/DolphinMind_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch transformers bitsandbytes accelerate rich psutil \
    sentence-transformers faiss-cpu rouge-score scikit-learn gradio

print("all dependencies installed")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 114.7 MB/s eta 0:00:00
all dependencies installed


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import psutil
import time
from rich.console import Console
from rich.table import Table
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import gc
import gradio as gr
import urllib.request

console = Console()
console.print("imports successful")

imports successful

In [3]:
def profile():
    ram = psutil.virtual_memory().used / 1e9
    console.print(f"RAM Used: {ram:.2f} GB")

In [4]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

console.print(f"Loading {model_name} in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

console.print("model loaded")
profile()

Loading Qwen/Qwen2.5-3B-Instruct in 4-bit...

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model loaded

RAM Used: 5.39 GB

In [5]:
console.print("Loading embedding model...")
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embedding_model = embedding_model.cpu()
console.print("embedding model loaded")
profile()

Loading embedding model...

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding model loaded

RAM Used: 3.71 GB

In [6]:
def ask(prompt, max_new_tokens=150):
    """Generate answer using proper chat template."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer the question based on the provided context. Be concise and direct."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the assistant's response
    if "assistant" in generated_text:
        answer = generated_text.split("assistant")[-1].strip()
    else:
        answer = generated_text.strip()

    return answer

# Cell 7: Baseline Truncated Method
def baseline_truncated(context, query, max_tokens=4000):
    """Simply truncate context to fit in window."""
    context_tokens = tokenizer.encode(context)[:max_tokens]
    truncated_context = tokenizer.decode(context_tokens, skip_special_tokens=True)

    prompt = f"Context: {truncated_context}\n\nQuestion: {query}\n\nAnswer:"
    response = ask(prompt)

    return response

In [7]:
def baseline_truncated(context, query, max_tokens=4000):
    """Simply truncate context to fit in window."""
    context_tokens = tokenizer.encode(context)[:max_tokens]
    truncated_context = tokenizer.decode(context_tokens, skip_special_tokens=True)

    prompt = f"Context: {truncated_context}\n\nQuestion: {query}\n\nAnswer:"
    response = ask(prompt)

    return response

In [8]:
def baseline_dolphinmind(context, query, chunk_size=512, overlap=128, top_k=5):
    """
    DolphinMind: Overlapping semantic chunks with chronological reconstruction.
    """
    # Stage 1: Context length check (with warning suppression for long docs)
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore')
        context_tokens = tokenizer.encode(context + query)

    if len(context_tokens) <= 4000:
        return baseline_truncated(context, query)

    # Stage 2: Sentence-aware chunking with overlap
    sentences = re.split(r'(?<=[.!?])\s+', context)

    chunks = []
    current_chunk = []
    current_length = 0

    i = 0
    while i < len(sentences):
        sentence = sentences[i]
        sentence_len = len(sentence.split())

        current_chunk.append(sentence)
        current_length += sentence_len

        if current_length >= chunk_size:
            text_chunk = ' '.join(current_chunk)
            chunks.append(text_chunk)

            overlap_buffer = []
            overlap_len = 0
            back_idx = i

            while back_idx >= 0 and overlap_len < overlap:
                overlap_buffer.insert(0, sentences[back_idx])
                overlap_len += len(sentences[back_idx].split())
                back_idx -= 1

            current_chunk = overlap_buffer
            current_length = overlap_len

        i += 1

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    # Stage 3: Encode chunks and build FAISS index
    embeddings = embedding_model.encode(chunks, show_progress_bar=False, batch_size=32)

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    faiss.normalize_L2(embeddings)
    index.add(embeddings)

    # Stage 4: Semantic retrieval
    query_embedding = embedding_model.encode([query])
    faiss.normalize_L2(query_embedding)
    distances, indices = index.search(query_embedding, top_k)

    # Stage 5: Chronological reconstruction
    retrieved_indices = sorted(indices[0])

    relevant_chunks = []
    for idx in retrieved_indices:
        if idx != -1 and idx < len(chunks):
            relevant_chunks.append(chunks[idx])

    combined = '\n\n'.join(relevant_chunks)

    # Stage 6: Token budget check
    retrieved_tokens = len(tokenizer.encode(combined))

    if retrieved_tokens > 3500:
        context_tokens = tokenizer.encode(combined)[:3500]
        combined = tokenizer.decode(context_tokens, skip_special_tokens=True)

    # Stage 7: Generate answer
    prompt = f"Context: {combined}\n\nQuestion: {query}\n\nAnswer:"
    response = ask(prompt)

    return response

console.print("inference functions defined")

inference functions defined

In [9]:
print("\nLoading text from Project Gutenberg...")
import urllib.request
import warnings

url = "https://www.gutenberg.org/files/14833/14833-0.txt"

try:
    with urllib.request.urlopen(url) as response:
        long_context = response.read().decode('utf-8')

    start_marker = "*** START OF"
    end_marker = "*** END OF"

    if start_marker in long_context:
        long_context = long_context.split(start_marker)[1]
    if end_marker in long_context:
        long_context = long_context.split(end_marker)[0]

    print("Downloaded Varney the Vampire")
    print(f"Words: {len(long_context.split()):,}")

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore')
        token_count = len(tokenizer.encode(long_context))
    print(f"Estimated tokens: {token_count:,}")

    story_marker = "The solemn tones of an old cathedral clock"
    idx = long_context.find(story_marker)

    if idx != -1:
        story_start = long_context[idx:]
        paragraphs = [p.strip() for p in story_start.split('\n\n') if len(p.strip()) > 50]
        sample_preview = '\n\n'.join(paragraphs[:5])
    else:
        story_start = 15000
        story_text = long_context[story_start:]
        paragraphs = [p.strip() for p in story_text.split('\n\n') if len(p.strip()) > 50]
        sample_preview = '\n\n'.join(paragraphs[:5])

except Exception as e:
    print(f"Failed to download: {e}")
    print("Using fallback text")
    long_context = """Machine learning is a branch of artificial intelligence.""" * 1000
    sample_preview = long_context[:2000]

test_queries = [
    "Who is Sir Francis Varney and what makes him terrifying?",
    "What happens to Flora Bannerworth in the story?",
    "How does the story end for Varney?"
]

print(f"Created {len(test_queries)} test queries")


Loading text from Project Gutenberg...
Downloaded Varney the Vampire
Words: 329,160


Token indices sequence length is longer than the specified maximum sequence length for this model (453222 > 131072). Running this sequence through the model will result in indexing errors


Estimated tokens: 453,222
Created 3 test queries


In [16]:
def process_single(context, query, method):
    """Process query with selected method."""
    if not context or not query:
        return "error: please provide both context and query.", ""

    token_count = len(tokenizer.encode(context))
    start_time = time.time()

    if method == "dolphinmind (semantic rag)":
        answer = baseline_dolphinmind(context, query)
        elapsed = time.time() - start_time
        estimated_tokens = min(3500, token_count)

        stats = f"""processing time: {elapsed:.2f}s
tokens used: {estimated_tokens:,} / {token_count:,} ({100*estimated_tokens/token_count:.1f}%)
ram usage: {psutil.virtual_memory().used / 1e9:.2f} gb
token reduction: {100*(token_count - estimated_tokens)/token_count:.1f}%

note: only processes the most relevant sections"""

    else:
        answer = baseline_truncated(context, query)
        elapsed = time.time() - start_time
        tokens_used = min(4000, token_count)
        lost_tokens = max(0, token_count - 4000)

        stats = f"""processing time: {elapsed:.2f}s
tokens used: {tokens_used:,} / {token_count:,}
ram usage: {psutil.virtual_memory().used / 1e9:.2f} gb
tokens lost: {lost_tokens:,} ({100*lost_tokens/token_count:.1f}% of document discarded)

note: can only see first ~4000 tokens of document"""

    return answer, stats


def compare_methods(context, query):
    """Run both methods side-by-side."""
    if not context or not query:
        return "error: please provide both context and query.", "", "", ""

    token_count = len(tokenizer.encode(context))

    # Run baseline
    start_time = time.time()
    baseline_answer = baseline_truncated(context, query)
    baseline_time = time.time() - start_time
    baseline_tokens = min(4000, token_count)

    # Run dolphinmind
    start_time = time.time()
    dolphin_answer = baseline_dolphinmind(context, query)
    dolphin_time = time.time() - start_time
    dolphin_tokens = min(3500, token_count)

    baseline_stats = f"""time: {baseline_time:.2f}s
tokens: {baseline_tokens:,}
ram: {psutil.virtual_memory().used / 1e9:.2f} gb"""

    dolphin_stats = f"""time: {dolphin_time:.2f}s
tokens: {dolphin_tokens:,} ({100*dolphin_tokens/token_count:.1f}% of doc)
ram: {psutil.virtual_memory().used / 1e9:.2f} gb
token reduction: {100*(token_count - dolphin_tokens)/token_count:.1f}%"""

    return baseline_answer, baseline_stats, dolphin_answer, dolphin_stats

console.print("gradio functions defined")

gradio functions defined

In [18]:
dolphin_cyan = "#23D5D5"
dolphin_dark = "#000000"

custom_css = """
.gradio-container {
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
}
.contain {
    max-width: 1200px;
}
body {
    background: linear-gradient(135deg, #000000 0%, #001a1a 100%);
}
"""

theme = gr.themes.Soft(
    primary_hue=gr.themes.colors.cyan,
    secondary_hue=gr.themes.colors.cyan,
).set(
    button_primary_background_fill=dolphin_cyan,
    button_primary_background_fill_hover="#1ab8b8",
    button_primary_text_color=dolphin_dark,
)

with gr.Blocks(theme=theme, css=custom_css, title="dolphinmind demo") as demo:

    gr.Markdown("""
    # dolphinmind: efficient long-context processing

    semantic retrieval beats raw context window size.
    process 100k+ token documents with 94.5% fewer tokens than native long-context models.

    ## quick facts
    - better accuracy than smollm3 @ 64k context (+20% rouge-l)
    - 94.5% fewer tokens processed per query
    - ~0.5 gb ram overhead (embedding model + faiss index)
    - runs on google colab free tier
    """)

    with gr.Tab("side-by-side comparison"):
        gr.Markdown("""### compare dolphinmind vs baseline on the same question

**baseline (truncated):** uses the same qwen2.5-3b model but only sees first ~4000 tokens of document

**dolphinmind:** uses semantic retrieval to find relevant sections from entire document

this shows the actual improvement from our approach.""")

        with gr.Row():
            with gr.Column():
                context_compare = gr.Textbox(
                    label="document",
                    lines=8,
                    value=sample_preview + f"\n\n[full document loaded: {len(long_context.split()):,} words, {len(tokenizer.encode(long_context)):,} tokens]"
                )

                query_compare = gr.Textbox(
                    label="question",
                    value=test_queries[0]
                )

                compare_btn = gr.Button("compare methods", variant="primary")

        with gr.Row():
            with gr.Column():
                gr.Markdown("### baseline (truncated context)")
                gr.Markdown("*same qwen2.5-3b model, only sees first 4000 tokens*")
                gr.Markdown("**cannot access information beyond first ~4000 tokens**")
                baseline_answer = gr.Textbox(label="answer", lines=6)
                baseline_stats = gr.Textbox(label="statistics", lines=5)

            with gr.Column():
                gr.Markdown("###dolphinmind (our approach)")
                gr.Markdown("*same model + semantic retrieval across full document*")
                gr.Markdown(" **searches entire document, uses top 5 most relevant chunks (~3500 tokens)**")
                dolphin_answer = gr.Textbox(label="answer", lines=6)
                dolphin_stats = gr.Textbox(label="statistics", lines=5)

        compare_btn.click(
            fn=compare_methods,
            inputs=[context_compare, query_compare],
            outputs=[baseline_answer, baseline_stats, dolphin_answer, dolphin_stats]
        )

    with gr.Tab("single method test"):
        gr.Markdown("### test individual methods on your own questions")

        with gr.Row():
            with gr.Column():
                context_input = gr.Textbox(
                    label="document (paste long text)",
                    placeholder="paste your document here...",
                    lines=10,
                    value=sample_preview + f"\n\n[full document loaded: {len(long_context.split()):,} words, {len(tokenizer.encode(long_context)):,} tokens]"
                )

                query_input = gr.Textbox(
                    label="your question",
                    placeholder="what would you like to know?",
                    value=test_queries[0]
                )

                method_choice = gr.Radio(
                    choices=[
                        "dolphinmind (semantic rag)",
                        "baseline (truncated context)"
                    ],
                    label="method",
                    value="dolphinmind (semantic rag)"
                )

                run_btn = gr.Button("run query", variant="primary")

            with gr.Column():
                answer_output = gr.Textbox(
                    label="answer",
                    lines=8
                )

                stats_output = gr.Textbox(
                    label="statistics",
                    lines=8
                )

        run_btn.click(
            fn=process_single,
            inputs=[context_input, query_input, method_choice],
            outputs=[answer_output, stats_output]
        )

    with gr.Tab("benchmark results"):
        gr.Markdown("""
        ## experimental results
        **dataset:** varney the vampire (329k words, ~453k tokens)

        | method | rouge-l | tokens/query | time (s) | approach |
        |--------|---------|--------------|----------|----------|
        | **dolphinmind** | **0.185** | **3,500** | 22.5 | semantic rag |
        | smollm3 (64k) | 0.154 | 64,085 | 13.0 | native long context |
        | rlm-tools | 0.154 | 2,100 | 7.2 | tool calling |
        | truncated | 0.146 | 4,000 | 7.8 | baseline |
        | naive chunking | 0.135 | 2,800 | 6.3 | tf-idf retrieval |

        ### key findings

        1. dolphinmind achieves highest accuracy (0.185 rouge-l)
           - beats baseline (truncated) by +26.7%
           - beats smollm3 @ 64k context by +20.1%
           - beats all other qwen2.5-3b methods

        2. 94.5% token reduction vs native 64k context
           - 3,500 tokens/query vs 64,085
           - 99.2% reduction vs full document

        3. more context does not equal better performance
           - smollm3 @ 128k: 0.132 rouge-l (14% worse than 64k)
           - semantic retrieval beats raw context window size

        4. minimal ram overhead
           - base model: 5.28 gb
           - dolphinmind overhead: ~0.46 gb
           - peak ram: 5.84 gb (fits colab free tier)
        """)

    with gr.Tab("how it works"):
        gr.Markdown("""
        ## dolphinmind pipeline

        ### what makes dolphinmind different?

        **baseline approach (truncated context):**
        - uses qwen2.5-3b-instruct directly
        - can only see first ~4000 tokens of document
        - if answer is in second half of book, it fails
        - rouge-l score: 0.146

        **dolphinmind approach (our contribution):**
        - same qwen2.5-3b-instruct model
        - adds semantic retrieval to find relevant sections
        - can access information from anywhere in 450k+ token document
        - rouge-l score: 0.185 (+26.7% improvement)

        ### 4-stage process

        **stage 1: context length check**
        - if document fits in native window (4k tokens) -> use standard inference
        - if too long -> proceed to chunking

        **stage 2: semantic chunking**
        - split by sentence boundaries (preserves meaning)
        - sliding window with overlap (maintains continuity)
        - default: 512-word chunks with 128-word overlap

        **stage 3: semantic retrieval**
        - encode chunks with all-minilm-l6-v2 (384-dim embeddings)
        - build faiss index (cpu-offloaded, ~0.3 gb)
        - retrieve top-5 most relevant chunks for query

        **stage 4: generation**
        - reconstruct chunks in chronological order
        - truncate to 3500 tokens if needed
        - generate answer with qwen2.5-3b-instruct

        ### why it works

        - overlap preserves discourse continuity
        - neural embeddings better than tf-idf for semantic matching
        - chronological reconstruction maintains narrative flow
        - cpu-offloaded architecture reduces gpu memory

        ### technical stack
        - model: qwen2.5-3b-instruct (4-bit quantized)
        - embeddings: sentence-transformers/all-minilm-l6-v2
        - retrieval: faiss (indexflatip)
        - framework: transformers, pytorch
        """)

# Launch
console.print("\nlaunching gradio interface...")
demo.queue()
demo.launch(share=True, debug=True)

/tmp/ipython-input-1996751773.py:25: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=custom_css, title="dolphinmind demo") as demo:
/tmp/ipython-input-1996751773.py:25: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=custom_css, title="dolphinmind demo") as demo:


launching gradio interface...

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bb57ab185fb1b4f468.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 0.0.0.0:7860 <> https://b7796277bc61124a00.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://bb57ab185fb1b4f468.gradio.live
